In [1]:
import os
print(os.environ.get("JAVA_HOME"))

C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot\


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("EmployeeAnalysis") \
    .getOrCreate()

print("Spark Started")
print("Spark Version:", spark.version)

Spark Started
Spark Version: 4.1.2


In [3]:
import os
print(os.getcwd())

C:\Users\lenovo


In [4]:
csv_data = """id,name,age,salary,department,region
1,Amit,25,50000,IT,North
2,Riya,30,60000,HR,South
3,Rahul,28,55000,IT,North
4,Riya,30,60000,HR,South
5,Neha,,45000,Finance,West
6,Karan,35,,IT,East
7,,29,52000,Marketing,South
"""

with open("employees.csv", "w") as file:
    file.write(csv_data)

print("employees.csv created successfully")

employees.csv created successfully


In [5]:
import os
print("employees.csv" in os.listdir())

True


In [6]:
df = spark.read.csv(
    "employees.csv",
    header=True,
    inferSchema=True
)

df.show()

+---+-----+----+------+----------+------+
| id| name| age|salary|department|region|
+---+-----+----+------+----------+------+
|  1| Amit|  25| 50000|        IT| North|
|  2| Riya|  30| 60000|        HR| South|
|  3|Rahul|  28| 55000|        IT| North|
|  4| Riya|  30| 60000|        HR| South|
|  5| Neha|NULL| 45000|   Finance|  West|
|  6|Karan|  35|  NULL|        IT|  East|
|  7| NULL|  29| 52000| Marketing| South|
+---+-----+----+------+----------+------+



In [7]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- region: string (nullable = true)



In [8]:
df.count()

7

In [9]:
df_no_duplicates = df.dropDuplicates()
df_no_duplicates.show()

+---+-----+----+------+----------+------+
| id| name| age|salary|department|region|
+---+-----+----+------+----------+------+
|  3|Rahul|  28| 55000|        IT| North|
|  2| Riya|  30| 60000|        HR| South|
|  4| Riya|  30| 60000|        HR| South|
|  1| Amit|  25| 50000|        IT| North|
|  6|Karan|  35|  NULL|        IT|  East|
|  5| Neha|NULL| 45000|   Finance|  West|
|  7| NULL|  29| 52000| Marketing| South|
+---+-----+----+------+----------+------+



In [10]:
print("Before:", df.count())
print("After:", df_no_duplicates.count())

Before: 7
After: 7


In [11]:
df_no_duplicates = df.dropDuplicates(
    ["name","age","salary","department","region"]
)

In [12]:
df_no_duplicates.show()

+---+-----+----+------+----------+------+
| id| name| age|salary|department|region|
+---+-----+----+------+----------+------+
|  1| Amit|  25| 50000|        IT| North|
|  6|Karan|  35|  NULL|        IT|  East|
|  3|Rahul|  28| 55000|        IT| North|
|  2| Riya|  30| 60000|        HR| South|
|  7| NULL|  29| 52000| Marketing| South|
|  5| Neha|NULL| 45000|   Finance|  West|
+---+-----+----+------+----------+------+



In [14]:
df_no_duplicates.show()

+---+-----+----+------+----------+------+
| id| name| age|salary|department|region|
+---+-----+----+------+----------+------+
|  1| Amit|  25| 50000|        IT| North|
|  6|Karan|  35|  NULL|        IT|  East|
|  3|Rahul|  28| 55000|        IT| North|
|  2| Riya|  30| 60000|        HR| South|
|  7| NULL|  29| 52000| Marketing| South|
|  5| Neha|NULL| 45000|   Finance|  West|
+---+-----+----+------+----------+------+



In [15]:
df_clean = df_no_duplicates.fillna({
    "age": 0,
    "salary": 0,
    "name": "Unknown"
})

In [16]:
df_clean.show()

+---+-------+---+------+----------+------+
| id|   name|age|salary|department|region|
+---+-------+---+------+----------+------+
|  1|   Amit| 25| 50000|        IT| North|
|  6|  Karan| 35|     0|        IT|  East|
|  3|  Rahul| 28| 55000|        IT| North|
|  2|   Riya| 30| 60000|        HR| South|
|  7|Unknown| 29| 52000| Marketing| South|
|  5|   Neha|  0| 45000|   Finance|  West|
+---+-------+---+------+----------+------+



In [17]:
df_clean.select(
    "name",
    "age",
    "salary"
).show()

+-------+---+------+
|   name|age|salary|
+-------+---+------+
|   Amit| 25| 50000|
|  Rahul| 28| 55000|
|   Riya| 30| 60000|
|  Karan| 35|     0|
|Unknown| 29| 52000|
|   Neha|  0| 45000|
+-------+---+------+



In [18]:
df_clean.show()

+---+-------+---+------+----------+------+
| id|   name|age|salary|department|region|
+---+-------+---+------+----------+------+
|  1|   Amit| 25| 50000|        IT| North|
|  6|  Karan| 35|     0|        IT|  East|
|  3|  Rahul| 28| 55000|        IT| North|
|  2|   Riya| 30| 60000|        HR| South|
|  7|Unknown| 29| 52000| Marketing| South|
|  5|   Neha|  0| 45000|   Finance|  West|
+---+-------+---+------+----------+------+



In [20]:
df_clean.filter(
    (df_clean.age >= 25) &
    (df_clean.age <= 35)
).show()

+---+-------+---+------+----------+------+
| id|   name|age|salary|department|region|
+---+-------+---+------+----------+------+
|  1|   Amit| 25| 50000|        IT| North|
|  6|  Karan| 35|     0|        IT|  East|
|  3|  Rahul| 28| 55000|        IT| North|
|  2|   Riya| 30| 60000|        HR| South|
|  7|Unknown| 29| 52000| Marketing| South|
+---+-------+---+------+----------+------+



In [21]:
df_clean.filter(
    df_clean.department == "IT"
).show()

+---+-----+---+------+----------+------+
| id| name|age|salary|department|region|
+---+-----+---+------+----------+------+
|  1| Amit| 25| 50000|        IT| North|
|  6|Karan| 35|     0|        IT|  East|
|  3|Rahul| 28| 55000|        IT| North|
+---+-----+---+------+----------+------+



In [22]:
df_clean.filter(
    df_clean.region == "North"
).show()

+---+-----+---+------+----------+------+
| id| name|age|salary|department|region|
+---+-----+---+------+----------+------+
|  1| Amit| 25| 50000|        IT| North|
|  3|Rahul| 28| 55000|        IT| North|
+---+-----+---+------+----------+------+



In [23]:
from pyspark.sql.functions import *

In [24]:
df_clean.select(
    count("*").alias("Total Employees")
).show()

+---------------+
|Total Employees|
+---------------+
|              6|
+---------------+



In [25]:
df_clean.select(
    sum("salary").alias("Total Salary")
).show()

+------------+
|Total Salary|
+------------+
|      262000|
+------------+



In [26]:
df_clean.select(
    avg("salary").alias("Average Salary")
).show()

+------------------+
|    Average Salary|
+------------------+
|43666.666666666664|
+------------------+



In [27]:
df_clean.select(
    min("salary").alias("Minimum Salary")
).show()

+--------------+
|Minimum Salary|
+--------------+
|             0|
+--------------+



In [28]:
df_clean.select(
    max("salary").alias("Maximum Salary")
).show()

+--------------+
|Maximum Salary|
+--------------+
|         60000|
+--------------+



In [29]:
df_clean.groupBy(
    "department"
).count().show()

+----------+-----+
|department|count|
+----------+-----+
|        HR|    1|
|   Finance|    1|
| Marketing|    1|
|        IT|    3|
+----------+-----+



In [30]:
df_clean.groupBy(
    "department"
).agg(
    avg("salary").alias("avg_salary")
).show()

+----------+----------+
|department|avg_salary|
+----------+----------+
|        HR|   60000.0|
|   Finance|   45000.0|
| Marketing|   52000.0|
|        IT|   35000.0|
+----------+----------+



In [31]:
df_clean.groupBy(
    "department"
).agg(
    avg("salary").alias("avg_salary")
).filter(
    col("avg_salary") > 50000
).show()

+----------+----------+
|department|avg_salary|
+----------+----------+
|        HR|   60000.0|
| Marketing|   52000.0|
+----------+----------+



In [32]:
df_clean.filter(df_clean.age > 25)

DataFrame[id: int, name: string, age: int, salary: int, department: string, region: string]

In [33]:
df_clean.groupBy("department").count()

DataFrame[department: string, count: bigint]

In [35]:
df_schema = df_clean.withColumnRenamed(
    "salary",
    "employee_salary"
)

In [36]:
df_schema.show()

+---+-------+---+---------------+----------+------+
| id|   name|age|employee_salary|department|region|
+---+-------+---+---------------+----------+------+
|  1|   Amit| 25|          50000|        IT| North|
|  6|  Karan| 35|              0|        IT|  East|
|  3|  Rahul| 28|          55000|        IT| North|
|  2|   Riya| 30|          60000|        HR| South|
|  7|Unknown| 29|          52000| Marketing| South|
|  5|   Neha|  0|          45000|   Finance|  West|
+---+-------+---+---------------+----------+------+



In [37]:
df_schema = df_schema.withColumn(
    "employee_salary",
    col("employee_salary").cast("double")
)

In [38]:
df_schema.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = false)
 |-- age: integer (nullable = false)
 |-- employee_salary: double (nullable = false)
 |-- department: string (nullable = true)
 |-- region: string (nullable = true)



In [39]:
df_standardized = df_schema.withColumn(
    "department",
    upper(col("department"))
)

In [40]:
df_standardized.show()

+---+-------+---+---------------+----------+------+
| id|   name|age|employee_salary|department|region|
+---+-------+---+---------------+----------+------+
|  1|   Amit| 25|        50000.0|        IT| North|
|  6|  Karan| 35|            0.0|        IT|  East|
|  3|  Rahul| 28|        55000.0|        IT| North|
|  2|   Riya| 30|        60000.0|        HR| South|
|  7|Unknown| 29|        52000.0| MARKETING| South|
|  5|   Neha|  0|        45000.0|   FINANCE|  West|
+---+-------+---+---------------+----------+------+



In [41]:
from pyspark.sql.functions import *

pipeline_result = (
    df
    .dropDuplicates(
        ["name","age","salary","department","region"]
    )
    .fillna({
        "age": 0,
        "salary": 0,
        "name": "Unknown"
    })
    .filter(
        (col("age") >= 25) &
        (col("age") <= 35)
    )
    .groupBy("department")
    .agg(
        count("*").alias("employee_count"),
        avg("salary").alias("avg_salary"),
        sum("salary").alias("total_salary"),
        min("salary").alias("min_salary"),
        max("salary").alias("max_salary")
    )
)

In [42]:
pipeline_result.show()

+----------+--------------+----------+------------+----------+----------+
|department|employee_count|avg_salary|total_salary|min_salary|max_salary|
+----------+--------------+----------+------------+----------+----------+
|        HR|             1|   60000.0|       60000|     60000|     60000|
| Marketing|             1|   52000.0|       52000|     52000|     52000|
|        IT|             3|   35000.0|      105000|         0|     55000|
+----------+--------------+----------+------------+----------+----------+

